# Task 5.3 — Evidence Verification

## Objective
Verify that claims in the Task 5.2 assessment are supported by the exact tool results recorded in its execution trace. The verifier must check more than whether a citation exists.

This notebook implements the current `VERIFIER_SPEC.md` requirements:
1. Evidence linkage.
2. Numerical agreement.
3. Direction and comparison agreement.
4. Entity grounding.

It preserves an untouched copy of the raw assessment, removes unsupported claims only from a separate verified copy, and reports the block rate and reasons.

## 1. Imports and load the Task 5.2 handoff
Run Task 5.2 first. This notebook will not invent a substitute assessment if the handoff is missing.

In [ ]:
from pathlib import Path
import json, copy
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

handoff_path = Path("task_outputs") / "task_5_2_run.json"
if not handoff_path.exists():
    raise FileNotFoundError(
        "Task 5.2 handoff not found. Complete a real Task 5.2 run first; "
        "do not use a demonstration/stub assessment as project evidence."
    )

handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
if handoff.get("execution_mode") != "REAL_TOOLS_AND_LLM":
    raise RuntimeError(
        "This handoff is not marked as a real tool + LLM run. "
        "Task 5.3 will not treat demonstration outputs as project evidence."
    )

raw_assessment = copy.deepcopy(handoff["assessment_raw"])
verified_assessment = copy.deepcopy(handoff["assessment_raw"])
trace = copy.deepcopy(handoff["trace"])

print("Source run:", handoff.get("run_id"))
print("Firm/year:", handoff.get("firm_id"), handoff.get("feature_year"))
print("Tool calls:", handoff.get("tool_calls_used"))
print("Raw evidence items:", len(raw_assessment.get("evidence", [])))
print("Raw contradictory items:", len(raw_assessment.get("contradictory_evidence", [])))

## 2. Verifier logic
The code below follows the reason categories in `VERIFIER_SPEC.md`. It uses the cited tool results and their arguments, not outside assumptions. Numeric verification is deliberately conservative: unsupported numbers block a claim.

In [ ]:
"""Task 5.3: evidence verifier (linkage, numeric, direction, entity checks).

The verifier is intentionally conservative. It checks claims in the evidence
and contradictory_evidence sections and removes any item that fails a check.
"""
from __future__ import annotations
import copy, re
from collections import Counter

REASONS = (
    "missing_or_unknown_ids",
    "numeric_claim_unsupported",
    "direction_contradicts_result",
    "entity_not_in_results",
)

NUMBER_RE = re.compile(r"(?:(?<=FY)|(?<![A-Za-z]))[-+]?\d[\d,]*(?:\.\d+)?%?(?![A-Za-z])", re.IGNORECASE)
DIRECTION_TERMS = {
    "above": ("above", "higher", "greater", "exceeded", "increase", "increased", "rising"),
    "below": ("below", "lower", "less", "under", "decrease", "decreased", "falling"),
    "unchanged": ("unchanged", "same", "stable", "no change", "flat"),
}

def _tool_results(trace):
    found = {}
    for event in trace:
        if event.get("event") == "tool_call" and event.get("status") == "ok":
            result = event.get("result") or {}
            rid = result.get("result_id")
            if rid:
                found[str(rid)] = {
                    "tool": event.get("tool"),
                    "result": result,
                    "arguments": event.get("arguments", {}),
                }
    return found

def _flatten_numbers(obj):
    vals = set()
    if isinstance(obj, bool) or obj is None:
        return vals
    if isinstance(obj, (int, float)):
        vals.add(round(float(obj), 4))
    elif isinstance(obj, dict):
        for v in obj.values():
            vals |= _flatten_numbers(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            vals |= _flatten_numbers(v)
    return vals

def _claim_numbers(text):
    out = []
    for token in NUMBER_RE.findall(str(text)):
        pct = token.endswith("%")
        raw = token[:-1] if pct else token
        try:
            val = float(raw.replace(",", ""))
        except ValueError:
            continue
        # A percentage in prose is compared with the decimal equivalent too.
        out.append((round(val / 100 if pct else val, 4), round(val, 4), pct))
    return out

def _numeric_supported(claim, cited):
    available = set()
    for entry in cited:
        available |= _flatten_numbers(entry["result"])
        available |= _flatten_numbers(entry["arguments"])
    for normalized, literal, is_pct in _claim_numbers(claim):
        if is_pct:
            if normalized not in available and literal not in available:
                return False
        elif literal not in available:
            return False
    return True

def _direction_ok(claim, cited):
    text = str(claim).lower()
    for entry in cited:
        val = (entry["result"].get("value") or {})
        tool = entry.get("tool")
        if tool == "get_deviation" and isinstance(val, dict):
            direction = val.get("direction")
            if direction in DIRECTION_TERMS:
                for other, terms in DIRECTION_TERMS.items():
                    if other != direction and any(t in text for t in terms):
                        # Do not mistake "not above" for a positive above claim.
                        if not re.search(r"\bnot\s+(?:" + "|".join(map(re.escape, terms)) + r")\b", text):
                            return False
        if tool == "get_peer_cohort" and isinstance(val, dict):
            lift = val.get("lift")
            if isinstance(lift, (int, float)):
                if lift < 1 and any(w in text for w in ("riskier than peers", "higher than peers", "above peer average")):
                    return False
                if lift > 1 and any(w in text for w in ("safer than peers", "lower risk than peers", "below peer average")):
                    return False
    return True

def _entity_ok(claim, cited, output):
    text = str(claim).lower()
    known = set()
    # Assessment identifiers are legitimate only if at least one cited result
    # independently contains the same firm/year.
    assessment = output.get("assessment", {}) if isinstance(output, dict) else {}
    firm = str(assessment.get("firm_id", "")).lower()
    year = str(assessment.get("feature_year", ""))
    outcome_year = str(assessment.get("outcome_year", ""))
    if firm and firm in text:
        if not any(firm in json_text(e["result"]).lower() or firm in json_text(e["arguments"]).lower() for e in cited):
            return False
    for token in re.findall(r"\b(?:19|20)\d{2}\b", text):
        if token not in {str(x) for e in cited for x in _flatten_numbers(e["result"])}:
            # Accept fiscal years supplied as tool arguments, too.
            if not any(token in json_text(e["arguments"]) for e in cited):
                return False
    # Named industry / filing sections must be present in the cited result.
    for marker in ("business services", "item 7", "item 9a", "item 15", "auditor report"):
        if marker in text and not any(marker in json_text(e["result"]).lower() for e in cited):
            return False
    return True

def json_text(obj):
    import json
    return json.dumps(obj, default=str)

def verify(output: dict, trace: list[dict]) -> dict:
    """Verify in place and return a structured report."""
    if not isinstance(output, dict):
        raise TypeError("output must be a dictionary")
    results = _tool_results(trace)
    report = {"verified_items": 0, "blocked_items": 0, "block_rate": 0.0,
              "issues": [], "reasons": {r: 0 for r in REASONS}}
    for section in ("evidence", "contradictory_evidence"):
        items = output.get(section, [])
        kept = []
        for item in items if isinstance(items, list) else []:
            ids = item.get("tool_result_ids", []) if isinstance(item, dict) else []
            if not ids or not isinstance(ids, list) or any(str(i) not in results for i in ids):
                reason = "missing_or_unknown_ids"
                cited = []
            else:
                cited = [results[str(i)] for i in ids]
                claim = item.get("claim", "")
                if not _numeric_supported(claim, cited):
                    reason = "numeric_claim_unsupported"
                elif not _direction_ok(claim, cited):
                    reason = "direction_contradicts_result"
                elif not _entity_ok(claim, cited, output):
                    reason = "entity_not_in_results"
                else:
                    reason = None
            if reason:
                report["blocked_items"] += 1
                report["reasons"][reason] += 1
                report["issues"].append({
                    "section": section, "item": copy.deepcopy(item),
                    "reason": reason,
                    "cited_tool_results": copy.deepcopy(cited),
                })
            else:
                report["verified_items"] += 1
                kept.append(item)
        output[section] = kept
    total = report["verified_items"] + report["blocked_items"]
    report["block_rate"] = round(report["blocked_items"] / total, 4) if total else 0.0
    return report


## 3. Verify the actual assessment
`raw_assessment` remains unchanged. `verified_assessment` is the separate copy that the verifier is allowed to filter.

In [ ]:
verification_report = verify(verified_assessment, trace)

total_claims = verification_report["verified_items"] + verification_report["blocked_items"]
summary_df = pd.DataFrame([{
    "total_claims_checked": total_claims,
    "verified_claims": verification_report["verified_items"],
    "blocked_claims": verification_report["blocked_items"],
    "block_rate": verification_report["block_rate"],
}])
display(summary_df)

reason_df = pd.DataFrame([
    {"reason": reason, "count": count}
    for reason, count in verification_report["reasons"].items()
])
display(reason_df)

## 4. Worked examples of blocked claims
The specification requires at least one worked example. The table below displays the claim, reason, cited tool and raw tool result where available.

In [ ]:
worked_rows = []
for issue in verification_report["issues"]:
    item = issue.get("item") or {}
    cited = issue.get("cited_tool_results") or []
    worked_rows.append({
        "section": issue.get("section"),
        "claim": item.get("claim"),
        "reason": issue.get("reason"),
        "cited_tool": cited[0].get("tool") if cited else "No valid citation",
        "tool_result": json.dumps(cited[0].get("result"), default=str) if cited else "No successful cited result",
    })
worked_examples = pd.DataFrame(worked_rows)
if worked_examples.empty:
    print("No claims were blocked in this run. Review the test cases below to confirm the verifier blocks deliberately unsupported claims.")
else:
    display(worked_examples)

## 5. Visualise verification results
The first chart shows how many claims survived versus were blocked. The second shows why claims were blocked. A high block rate is not automatically good or bad: it depends on the quality of the draft claims and the verifier's ability to detect unsupported statements.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.bar(
    ["Verified", "Blocked"],
    [verification_report["verified_items"], verification_report["blocked_items"]],
    color=["#2D9D78", "#D85B56"]
)
ax.set_title("Evidence verification outcome")
ax.set_ylabel("Number of claims")
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), int(bar.get_height()),
            ha="center", va="bottom")
plt.tight_layout()
plt.show()

nonzero = reason_df[reason_df["count"] > 0]
if not nonzero.empty:
    ax = nonzero.set_index("reason")["count"].plot(
        kind="bar", figsize=(9,4), color="#3977A8"
    )
    ax.set_title("Blocked claims by verification reason")
    ax.set_xlabel("")
    ax.set_ylabel("Number of claims")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No claims were blocked; no reason-count chart is needed.")

display(Markdown(
    f"**Interpretation:** the verifier checked {total_claims} claims. "
    f"It retained {verification_report['verified_items']} and blocked "
    f"{verification_report['blocked_items']}. The block rate is "
    f"{verification_report['block_rate']:.1%}. Inspect the reason breakdown "
    "and worked examples before interpreting this as evidence of verifier quality."
))

## 6. Verifier behaviour tests
These are controlled unit tests, not real firm evidence. They test whether the verifier blocks a wrong number, a contradictory direction and an unknown result ID. The actual project assessment remains the one loaded from Task 5.2.

In [ ]:
def run_verifier_test(claim, tool_name, value, result_id="r_test"):
    test_output = {
        "assessment": {"firm_id": "001004", "feature_year": 2018, "outcome_year": 2019},
        "evidence": [{"claim": claim, "tool_result_ids": [result_id]}],
        "contradictory_evidence": [],
        "uncertainty": [],
        "recommended_checks": [],
    }
    test_trace = [{
        "event": "tool_call", "tool": tool_name, "status": "ok",
        "arguments": {"gvkey": "001004", "feature_year": 2018},
        "result": {"result_id": result_id, "value": value, "source": "unit-test fixture"},
    }]
    report = verify(test_output, test_trace)
    return report

tests = [
    ("ROA was 0.04.", "get_percentile", {"value": 0.04}),
    ("ROA was 0.04 and was above its own history.", "get_deviation", {"current": 0.04, "direction": "below"}),
    ("ROA was 0.99.", "get_percentile", {"value": 0.04}),
]
test_results = []
for claim, tool, value in tests:
    report = run_verifier_test(claim, tool, value)
    test_results.append({
        "claim": claim,
        "verified": report["verified_items"],
        "blocked": report["blocked_items"],
        "reason": next((k for k,v in report["reasons"].items() if v), "none"),
    })

# Explicit unknown-ID test
bad_id_output = {
    "assessment": {"firm_id": "001004", "feature_year": 2018},
    "evidence": [{"claim": "A claim with an unknown citation.", "tool_result_ids": ["missing_id"]}],
    "contradictory_evidence": [], "uncertainty": [], "recommended_checks": []
}
bad_id_report = verify(bad_id_output, [])
test_results.append({
    "claim": "Unknown result ID",
    "verified": bad_id_report["verified_items"],
    "blocked": bad_id_report["blocked_items"],
    "reason": next((k for k,v in bad_id_report["reasons"].items() if v), "none"),
})
test_df = pd.DataFrame(test_results)
display(test_df)

## 7. Save the verification outputs

In [ ]:
report_payload = {
    "task": "5.3",
    "source_run_id": handoff.get("run_id"),
    "firm_id": handoff.get("firm_id"),
    "feature_year": handoff.get("feature_year"),
    "raw_assessment": raw_assessment,
    "verified_assessment": verified_assessment,
    "verification_report": verification_report,
    "verifier_unit_tests": test_results,
}
out_path = Path("task_outputs") / "task_5_3_verification.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(report_payload, indent=2, default=str), encoding="utf-8")
print("Saved verification report:", out_path.resolve())

## 8. Interpretation and limitations

The verifier distinguishes **citation existence** from **claim support**. A valid result ID is necessary but not sufficient: the cited result must also support the claim's numbers, direction and named entities.

The block rate should be reported with the blocked-claim examples and reason breakdown. A low block rate may mean the agent produced well-supported claims, but it may also mean the verifier is too permissive. A high block rate may indicate poor draft quality, strict verification, or both.

The current specification treats qualitative language such as "materially" or "sharply" as later work. This notebook does not claim to verify those judgements. Filing-passage semantic verification is also separate and remains gated by the project's filing work.